# Naan Stop in Manhattan (experiment)

A copy of the Day 4 idea: Manhattan blown up with parks, streets and landmarks, and all five boroughs as a key map. The citywide notebook stays as it is. Streets and parks appear once `python scripts\\fetch_week1.py day4` has fetched them. Saves to `png_thumbnail/day04_experiment_manhattan.png` and leaves the gallery alone.

## 1 · Settings

In [ ]:
TITLE    = "Naan Stop\nin Manhattan"
OUTPUT   = "day04_experiment_manhattan.png"       # an experiment: saved beside the other posters, not added to the gallery
ANGLE    = 29                                     # turn the map so the avenues run straight up the poster

## 2 · Load

In [ ]:
import json
import geopandas as gpd, matplotlib.pyplot as plt, numpy as np, pandas as pd
import matplotlib.patheffects as pe
from shapely import affinity
from shapely.geometry import Point, Polygon, box
from sklearn.cluster import DBSCAN
from mapkit import poster
from mapkit.paths import PROCESSED, THUMBS

food = gpd.read_file(PROCESSED / "nyc_indian.geojson")
boroughs = gpd.read_file(PROCESSED / "nyc_boroughs.geojson")
def real_data(name):
    """Only real data gets drawn: no file, nothing on the map."""
    f = PROCESSED / name
    return gpd.read_file(f) if f.exists() else None
parks, streets = real_data("manhattan_parks.geojson"), real_data("manhattan_streets.geojson")

report = json.loads((PROCESSED / "fetch_report.json").read_text()).get("day4_manhattan", {})
boro_source = "NYC Dept. of City Planning" if (PROCESSED / "nyc_boroughs_source.txt").read_text() == "official" else "Click That Hood"
fetched = [f"Parks: {report.get('parks_source', 'NYC Parks')}." if parks is not None else "", f"Streets: {report.get('streets_source', 'NYC Street Centerline')}." if streets is not None else ""]
CREDITS = "Restaurants: NYC Health Dept. inspections, active Indian listings.\n" + (" ".join(x for x in fetched if x) + "\n" if any(fetched) else "") + f"Boroughs: {boro_source}. Landmarks placed by hand."

## 3 · Analysis, every number on the map comes from here

In [ ]:
# Same clusters as the citywide map, so the two versions agree
EPS_M, MIN_SIZE = 400, 5
m = food.to_crs(32618)
food["cluster"] = DBSCAN(eps=EPS_M, min_samples=MIN_SIZE).fit_predict(np.c_[m.geometry.x, m.geometry.y])

label = next(c for c in boroughs.columns if boroughs[c].astype(str).str.contains("Manhattan").any())
is_manhattan = boroughs[label].astype(str).str.contains("Manhattan")
island = boroughs[is_manhattan].to_crs(4326).union_all()
food["in_manhattan"] = food.boro == "Manhattan"     # the health department's own borough field

PLACES = {"Curry Hill": (-73.986, 40.745), "Midtown West": (-73.986, 40.762), "Midtown East": (-73.972, 40.755), "East Village": (-73.985, 40.727),
          "Greenwich Village": (-74.001, 40.730), "Tribeca": (-74.008, 40.716), "Upper West Side": (-73.966, 40.801)}
inside = food[(food.cluster >= 0) & food.in_manhattan]
sizes = inside.groupby("cluster").size().sort_values(ascending=False)
centers = inside.groupby("cluster").geometry.apply(lambda g: (g.x.mean(), g.y.mean()))
names = {c: min(PLACES, key=lambda p: (PLACES[p][0] - xy[0]) ** 2 + (PLACES[p][1] - xy[1]) ** 2) for c, xy in centers.items()}
assert len(set(names.values())) == len(names), "two clusters got the same name"
total, in_man, in_clusters = len(food), int(food.in_manhattan.sum()), int(sizes.sum())
print(f"Indian restaurants, citywide   {total}")
print(f"In Manhattan                   {in_man} ({in_man / total * 100:.0f}%)")
print(f"In Manhattan's clusters        {in_clusters} in {len(sizes)} clusters")
for c, n in sizes.items():
    print(f"  {names[c]:20} {n}")

## 4 · Map

In [ ]:
WATER, LAND, GROUND, PARK, STREET, CHILI, INK = "#ec9f12", "#fff3d6", "#f7cf7a", "#a9bd7d", "#e4d2a6", "#b3201f", "#3a120c"
fig, ax = poster.page(WATER)
ax.set_axis_off()

center = gpd.GeoSeries([Point(-73.975, 40.758)], crs=4326).to_crs(32618).iloc[0]
def turn(frame):
    """Rotate a layer so the avenues run up the page."""
    return frame.to_crs(32618).geometry.rotate(ANGLE, origin=center)

HALF_H = 7900
HALF_W = HALF_H * 0.8
cx, cy = center.x - 600, center.y + 300
xlim, ylim = (cx - HALF_W, cx + HALF_W), (cy - HALF_H, cy + HALF_H)

turn(boroughs[~is_manhattan]).plot(ax=ax, color=GROUND, lw=0, zorder=1)

# Under the key map the outlines turn light, the same see-through idea as the citywide legend
KEY_BOX = box(xlim[0] + (xlim[1] - xlim[0]) * 0.75, ylim[0], xlim[1], ylim[0] + (ylim[1] - ylim[0]) * 0.29)
others = turn(boroughs[~is_manhattan]).boundary.union_all()
gpd.GeoSeries([others.difference(KEY_BOX)], crs=32618).plot(ax=ax, color=INK, lw=0.5, zorder=1)
gpd.GeoSeries([others.intersection(KEY_BOX)], crs=32618).plot(ax=ax, color=INK, lw=0.4, alpha=0.3, zorder=1)
turn(boroughs[is_manhattan]).plot(ax=ax, color=LAND, edgecolor=INK, lw=1.0, zorder=1)
if parks is not None:
    turn(parks).plot(ax=ax, color=PARK, lw=0, zorder=2)

# Streets in two line weights: avenues heavier, cross streets fine
if streets is not None:
    is_avenue = streets.street.fillna("").str.contains(r"\bAV|\bAVE|AVENUE|BROADWAY|PARK AVE", case=False, regex=True)
    turn(streets[~is_avenue]).plot(ax=ax, color=STREET, lw=0.3, zorder=3)
    turn(streets[is_avenue]).plot(ax=ax, color="#c9b382", lw=0.65, zorder=3)
turn(boroughs[is_manhattan]).boundary.plot(ax=ax, color=INK, lw=1.0, zorder=4)

# Clusters: rings thinning as they spread, then every restaurant
for c in sizes.index:
    hull = turn(food[food.cluster == c]).union_all().convex_hull
    for grow, lw, alpha in [(45, 1.25, 1), (100, 0.75, 0.85), (170, 0.4, 0.7)]:
        gpd.GeoSeries([hull.buffer(grow)], crs=32618).plot(ax=ax, facecolor="none", edgecolor=CHILI, lw=lw, alpha=alpha, zorder=6)
    gpd.GeoSeries([hull.buffer(45)], crs=32618).plot(ax=ax, facecolor=CHILI, alpha=0.2, lw=0, zorder=5)
turn(food[(food.cluster < 0) & food.in_manhattan]).plot(ax=ax, color=INK, markersize=15, alpha=0.6, lw=0, zorder=7)
turn(inside).plot(ax=ax, color=CHILI, markersize=30, lw=0, zorder=8)
ax.set_xlim(xlim); ax.set_ylim(ylim)

def page_xy(lon, lat):
    return gpd.GeoSeries([affinity.rotate(gpd.GeoSeries([Point(lon, lat)], crs=4326).to_crs(32618).iloc[0], ANGLE, origin=center)]).iloc[0].coords[0]

# Cluster names out in the rivers, with a hairline back to each cluster
SIDES = {"Upper West Side": ("left", 0.655), "Midtown West": ("left", 0.5), "Curry Hill": ("right", 0.385), "Midtown East": ("right", 0.5),
         "East Village": ("right", 0.27), "Greenwich Village": ("left", 0.31), "Tribeca": ("left", 0.205)}
by_name = {names[c]: c for c in sizes.index}
for name, (side, fy) in SIDES.items():
    c = by_name[name]
    hx, hy = turn(food[food.cluster == c]).union_all().centroid.coords[0]
    y = ylim[0] + (ylim[1] - ylim[0]) * fy
    x = xlim[0] + (xlim[1] - xlim[0]) * (0.325 if side == "left" else (0.79 if name == "East Village" else 0.745))
    ax.plot([x, hx], [y, hy], color=INK, lw=0.55, zorder=9)
    ha = "right" if side == "left" else "left"
    gap = 45 if side == "right" else -45
    ax.text(x + gap, y + 80, name, color=INK, fontproperties=poster.font(poster.POPPINS_BOLD, 12.5), ha=ha, va="bottom", zorder=10)
    ax.text(x + gap, y + 60, f"{sizes[c]} restaurants", color=CHILI, fontproperties=poster.font(poster.POPPINS, 10.5), ha=ha, va="top", zorder=10)

# Landmarks, each kind drawn its own way. Positions placed by hand.
TOWN = dict(color=INK, lw=0, zorder=11)
KINDS = {"Towers and museums": dict(marker="*", s=85, **TOWN),
         "Transit hubs": dict(marker="o", s=38, facecolor=LAND, edgecolor=INK, linewidth=1.5, zorder=11),
         "Universities": dict(marker="s", s=26, **TOWN)}
LANDMARKS = {"Towers and museums": {"Empire State Building": (-73.9857, 40.7484), "One World Trade": (-74.0134, 40.7127), "The Met": (-73.9632, 40.7794), "Lincoln Center": (-73.9835, 40.7725)},
             "Transit hubs": {"Grand Central": (-73.9772, 40.7527), "Penn Station": (-73.9935, 40.7506)},
             "Universities": {"Columbia University": (-73.9626, 40.8075), "NYU": (-73.9965, 40.7295)}}
SIDE = {"Penn Station": "left"}
for kind, places in LANDMARKS.items():
    for name, (lon, lat) in places.items():
        assert island.buffer(0.001).contains(Point(lon, lat)), name
        x, y = page_xy(lon, lat)
        ax.scatter([x], [y], **KINDS[kind])
        left = SIDE.get(name) == "left"
        poster.halo_text(ax, x + (-70 if left else 70), y, name, color=INK, path=poster.POPPINS, size=8.8, halo=LAND, halo_width=2.2, ha="right" if left else "left", va="center", zorder=11)

# Public squares get a tinted shape and an italic name. Times Square is a simple hand-drawn bowtie along Broadway.
SQUARE = "#f0a53a"
tx, ty = page_xy(-73.9855, 40.7580)
bowtie = Polygon([(tx - 60, ty - 240), (tx + 40, ty - 240), (tx, ty), (tx + 60, ty + 210), (tx - 40, ty + 210), (tx, ty)])
gpd.GeoSeries([bowtie], crs=32618).plot(ax=ax, color=SQUARE, edgecolor=INK, lw=0.7, zorder=10)
GREEN = "#3f5320"
def italic(lon, lat, text, color=INK, size=11, rotation=0, dx=0, dy=0, ha="center"):
    x, y = page_xy(lon, lat)
    t = ax.text(x + dx, y + dy, text, color=color, fontproperties=poster.font(poster.OLD_ITALIC, size), rotation=rotation, ha=ha, va="center", zorder=11)
    t.set_path_effects([pe.withStroke(linewidth=2.2, foreground=LAND)])
ax.plot([tx, tx - 600], [ty, ty - 450], color=INK, lw=0.5, zorder=10)
italic(-73.9855, 40.7580, "Times Square", dx=-630, dy=-470, ha="right")
italic(-73.9906, 40.7359, "Union Square", dx=85, ha="left")
italic(-73.9973, 40.7308, "Washington Square", dx=-60, dy=400, ha="right")
if parks is not None:
    italic(-73.9654, 40.7829, "Central Park", color=GREEN, size=15, rotation=90)
    italic(-73.9705, 40.8025, "Riverside Park", color=GREEN, size=10, rotation=90, dx=-60, dy=-2200)

# Rivers, placed by position on the page
for text, px, py in [("Hudson River", 0.344, 0.60), ("East River", 0.736, 0.311)]:
    t = ax.text(px, py, text, color="#5a2a03", fontproperties=poster.font(poster.OLD_ITALIC, 15), rotation=90, ha="center", va="center", zorder=9, transform=ax.transAxes)
    t.set_path_effects([pe.withStroke(linewidth=2.4, foreground=WATER)])
ax.text(0.05, 0.42, "New Jersey", color="#7a3b06", fontproperties=poster.font(poster.OLD_ITALIC, 15), rotation=90, ha="center", va="center", zorder=9, transform=ax.transAxes)

fig.text(0.05, 0.968, "Naan Stop", color=INK, fontproperties=poster.font(poster.YATRA, 34), va="top")
fig.text(0.05, 0.925, "in Manhattan", color=CHILI, fontproperties=poster.font(poster.YATRA, 34), va="top")
fig.text(0.05, 0.875, f"{in_man} of New York's {total} Indian\nrestaurants are here. {in_clusters} sit in {len(sizes)}\ntight clusters, each restaurant a\nfive-minute walk from the next.", color=INK,
         fontproperties=poster.font(poster.POPPINS, 11), va="top", linespacing=1.4)

# What the symbols mean
y = 0.77
for label, style in KINDS.items():
    ax.scatter([xlim[0] + (xlim[1] - xlim[0]) * 0.058], [ylim[0] + (ylim[1] - ylim[0]) * y], **style)
    fig.text(0.078, y, label, color=INK, fontproperties=poster.font(poster.POPPINS, 9.5), va="center")
    y -= 0.022
fig.text(0.05, y, "Public squares", color=INK, fontproperties=poster.font(poster.OLD_ITALIC, 12), va="center")
if parks is not None:
    fig.text(0.05, y - 0.022, "Parks", color=GREEN, fontproperties=poster.font(poster.OLD_ITALIC, 12), va="center")
ax.set_xlim(xlim); ax.set_ylim(ylim)

# North has moved, so say so
arrow = fig.add_axes([0.905, 0.9, 0.06, 0.07]); arrow.set_axis_off(); arrow.set_xlim(-1, 1); arrow.set_ylim(-1, 1); arrow.set_aspect("equal")
a = np.radians(ANGLE)
arrow.annotate("", xy=(-np.sin(a) * 0.75, np.cos(a) * 0.75), xytext=(np.sin(a) * 0.55, -np.cos(a) * 0.55), arrowprops=dict(arrowstyle="-|>", color=INK, lw=1.1))
arrow.text(-np.sin(a) * 1.0, np.cos(a) * 1.0, "N", color=INK, fontproperties=poster.font(poster.POPPINS_BOLD, 10.5), ha="center", va="center")

# Key map: all five boroughs, true north, with the close-up marked
key = fig.add_axes([0.76, 0.065, 0.21, 0.21], facecolor="none"); key.set_axis_off()
boroughs.to_crs(3857).plot(ax=key, color=GROUND, edgecolor=INK, lw=0.35, zorder=1)
boroughs[is_manhattan].to_crs(3857).plot(ax=key, color=LAND, edgecolor=INK, lw=0.4, zorder=2)
food.to_crs(3857).plot(ax=key, color=CHILI, markersize=1.6, lw=0, zorder=3)
frame = affinity.rotate(box(xlim[0], ylim[0], xlim[1], ylim[1]), -ANGLE, origin=center)
gpd.GeoSeries([frame], crs=32618).to_crs(3857).plot(ax=key, facecolor="none", edgecolor=INK, lw=1.1, linestyle=(0, (2.2, 1.6)), zorder=4)
key.set_aspect("equal")
fig.text(0.965, 0.08, f"All five boroughs, {total} restaurants", color=INK, fontproperties=poster.font(poster.POPPINS_BOLD, 9), ha="right", va="center")
fig.text(0.05, 0.02, CREDITS, color="#4a2410", fontproperties=poster.font(poster.POPPINS, 7.6), va="bottom", linespacing=1.5)
fig.text(0.05, 0.078, "Map by Gautami Kasat", color=INK, fontproperties=poster.font(poster.POPPINS_BOLD, 10), va="bottom")

## 5 · Export

In [ ]:
out = THUMBS / OUTPUT
fig.savefig(out, dpi=100, facecolor=fig.get_facecolor())
print("saved", out)